# Piper Italian Voice — training → ONNX

Train a **Piper** voice from an LJSpeech dataset (`dataset/` with `wav/` + `metadata.csv`)
and export it to `model.onnx` + `model.onnx.json`.

**Runtime → Change runtime type → GPU** before you start.

## 1. GPU check

In [ ]:
!nvidia-smi

## 2. Upload the dataset

Zip the folder locally (`Compress-Archive -Path dataset -DestinationPath dataset.zip` on
Windows, or `zip -r dataset.zip dataset` elsewhere) and upload `dataset.zip` below.
Files survive the kernel restart in section 3.

In [ ]:
# Manual upload of dataset.zip
from google.colab import files
up = files.upload()  # choose dataset.zip
!unzip -q -o dataset.zip -d /content/
!ls /content/dataset && echo '---' && head -n 3 /content/dataset/metadata.csv

In [ ]:
# Alternative: Google Drive (put dataset.zip in your Drive)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp '/content/drive/MyDrive/dataset.zip' /content/dataset.zip
# !unzip -q -o /content/dataset.zip -d /content/

## 3. Install Piper (training)

Piper training needs **Python 3.10** + **pip<24.1** + specific older libraries. Steps:
**3a** (condacolab, restarts the kernel) → **3b** (Python 3.10 env) → **3c** (install).

After the restart in 3a, continue from **3b** (don't re-run 3a).

In [ ]:
# 3a) condacolab. >>> THIS CELL RESTARTS THE KERNEL ON ITS OWN <<<
#     After the restart, do NOT re-run it: go straight to 3b.
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
# 3b) create a dedicated Python 3.10 environment (run AFTER the restart)
!conda create -y -n p310 python=3.10
PY  = '/usr/local/envs/p310/bin/python'
PIP = '/usr/local/envs/p310/bin/pip'
# pip<24.1: recent pip rejects pytorch-lightning 1.7's dependency metadata
!{PIP} install -q 'pip<24.1'
!{PY} --version

In [ ]:
# 3c) install the training stack with known-working versions
!{PIP} install -q 'numpy==1.23.5' 'cython<1' 'torch==1.13.1' \
  'pytorch-lightning==1.7.7' 'torchmetrics==0.11.4' \
  'librosa==0.9.2' 'numba==0.56.4' 'onnxruntime' 'piper-phonemize~=1.1.0' \
  'setuptools<81' 'wheel' 'six'
%cd /content
!git clone -q https://github.com/rhasspy/piper.git
%cd /content/piper/src/python
!{PIP} install -q -e . --no-deps
# build monotonic_align using the env's Python 3.10
!PATH=/usr/local/envs/p310/bin:$PATH bash build_monotonic_align.sh
# final check (CUDA must be True)
!{PY} -c "import torch, pytorch_lightning as pl, piper_phonemize; print('torch', torch.__version__, '| cuda?', torch.cuda.is_available(), '| pl', pl.__version__)"

In [ ]:
# 3c-bis) RUN ONLY IF 'cuda?' was False above: force the GPU build of torch
# !{PIP} install -q 'torch==1.13.1+cu117' --index-url https://download.pytorch.org/whl/cu117

## 4. Preprocess (LJSpeech → training cache)

Resamples to 22050 Hz, phonemizes with espeak-ng (`--language it`) and prepares the training files.

In [ ]:
%cd /content/piper/src/python
!{PY} -m piper_train.preprocess \
  --language it \
  --input-dir /content/dataset \
  --output-dir /content/training \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050
!ls /content/training

## 5. Download the base checkpoint (Spanish medium)

Piper checkpoints don't include Italian, so we start from Spanish `es_ES/davefx/medium`
(phonetically close; medium = 22050 Hz, matching the preprocess).

In [ ]:
# Spanish medium base checkpoint (URL verified on Hugging Face):
CKPT_URL = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/es/es_ES/davefx/medium/epoch%3D5629-step%3D1605020.ckpt'
!wget -q -O /content/base.ckpt "$CKPT_URL"
!ls -lh /content/base.ckpt

## 6. Training (fine-tuning)

For single-speaker, Piper needs `--resume_from_checkpoint` (`--resume_from_single_speaker_checkpoint`
is for multi-speaker models only). The base is at epoch **5629**: training CONTINUES from there,
so `--max_epochs` must be **> 5629** (10000 here).
If you hit OOM, lower `--batch-size` (e.g. 4).

In [ ]:
%cd /content/piper/src/python
!{PY} -m piper_train \
  --dataset-dir /content/training \
  --accelerator gpu --devices 1 \
  --batch-size 8 \
  --validation-split 0.0 --num-test-examples 0 \
  --max_epochs 10000 \
  --checkpoint-epochs 1 \
  --precision 32 \
  --resume_from_checkpoint /content/base.ckpt

In [ ]:
# (optional) monitor training
%load_ext tensorboard
%tensorboard --logdir /content/training/lightning_logs

## 7. Export to ONNX

In [ ]:
import glob, os
ckpts = sorted(glob.glob('/content/training/lightning_logs/version_*/checkpoints/*.ckpt'),
               key=os.path.getmtime)
# If you stopped training manually, the LAST checkpoint may be truncated (corrupt):
# use the second-to-last, which is fully written.
last_ckpt = ckpts[-2] if len(ckpts) >= 2 else ckpts[-1]
print('Using checkpoint:', last_ckpt)
!{PY} -m piper_train.export_onnx "{last_ckpt}" /content/model.onnx
!cp /content/training/config.json /content/model.onnx.json
!ls -lh /content/model.onnx /content/model.onnx.json

## 8. Download the model

In [ ]:
from google.colab import files
files.download('/content/model.onnx')
files.download('/content/model.onnx.json')

## Use it locally with Piper

```bash
echo "Ciao, sono la tua voce sintetica italiana." | piper --model model.onnx --output_file out.wav
```